# Blue Lobster — SST Data Pipeline
Downloads every NOAA OISST v2.1 daily file (1981-09-01 → present), extracts regional SST for three Atlantic Canadian sub-regions, aggregates to annual means, and writes a clean CSV for the habitat suitability model.

**Strategy:** download → extract → delete NC file. Peak disk use is one ~9 MB file at a time.  
**Resume:** safe to interrupt — a daily checkpoint CSV is flushed every 100 files.  
**Output:** `Data/sst_regions.csv` — columns: `region, year, mean_temp`


In [ ]:
import urllib.request
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import date, timedelta

DATA_DIR   = Path("Data")
CHECKPOINT = DATA_DIR / "sst_daily_checkpoint.csv"   # intermediate daily means (small, kept)
OUT_CSV    = DATA_DIR / "sst_regions.csv"             # final annual output

BASE_URL = (
    "https://www.ncei.noaa.gov/data/"
    "sea-surface-temperature-optimum-interpolation/v2.1/access/avhrr/"
)

# 1981 and 2026 excluded — only partial months of data available
START = date(1982, 1, 1)
END   = date(2025, 12, 31)

# Three regions — exact names required by John's scorer
REGIONS = {
    "Bay of Fundy":         {"lat": (44.0, 46.5), "lon": (-68.0, -63.5)},
    "Scotian Shelf":        {"lat": (42.5, 47.0), "lon": (-67.0, -57.0)},
    "Gulf of St. Lawrence": {"lat": (46.0, 51.0), "lon": (-66.0, -60.0)},
}

total_days = (END - START).days + 1
print(f"Date range : {START} → {END}  ({total_days:,} calendar days)")
print(f"Regions    : {list(REGIONS)}")
print(f"Checkpoint : {CHECKPOINT}")


## Helper functions

`remap_lons` converts OISST's 0–360° longitude axis to −180/180 so our bounding boxes work directly.  
`extract_regions` opens one file, pulls SST for each bounding box, returns daily regional means.  
The NC file is deleted immediately after extraction.


In [6]:
def make_url(d: date) -> str:
    ym  = d.strftime("%Y%m")
    ymd = d.strftime("%Y%m%d")
    return f"{BASE_URL}{ym}/oisst-avhrr-v02r01.{ymd}.nc"


def remap_lons(ds: xr.Dataset) -> xr.Dataset:
    lon_180 = (ds.lon.values + 180) % 360 - 180
    return ds.assign_coords(lon=lon_180).sortby("lon")


def extract_regions(nc_path: Path, date_str: str) -> list[dict]:
    """Open NC, compute mean SST per region, return list of row dicts."""
    ds  = xr.open_dataset(nc_path, mask_and_scale=True)
    ds  = remap_lons(ds)
    sst = ds["sst"].squeeze(drop=True)   # (lat, lon), °C

    rows = []
    for region, bbox in REGIONS.items():
        box  = sst.sel(lat=slice(bbox["lat"][0], bbox["lat"][1]),
                       lon=slice(bbox["lon"][0], bbox["lon"][1]))
        vals = box.values.ravel()
        ok   = vals[~np.isnan(vals)]
        rows.append({
            "date":   date_str,
            "region": region,
            "sst":    round(float(ok.mean()), 3) if len(ok) else np.nan,
        })

    ds.close()
    return rows


## Main pipeline — three progressive phases

| Phase | What it fetches | ~Files | After this, John has… |
|---|---|---|---|
| 1 | One file per year (July 15) | ~45 | 44-year trend, immediately |
| 2 | One file per month (the 15th) | ~540 | Seasonal shape per year |
| 3 | Every remaining day, **breadth-first** | ~16,300 | Full daily resolution |

Phase 3 breadth-first means: every year-month gets its 2nd day before any gets its 3rd, every year-month gets its 3rd before any gets its 4th, etc. Coverage stays uniform across all 44 years no matter when you interrupt.

Run the aggregate + export cells at any point for a snapshot CSV for John.


In [7]:
from calendar import monthrange

# ── Shared state ─────────────────────────────────────────────────────────────
if CHECKPOINT.exists():
    ckpt_df   = pd.read_csv(CHECKPOINT)
    processed = set(ckpt_df["date"])
    print(f"Resuming — {len(processed):,} dates already in checkpoint")
else:
    ckpt_df   = pd.DataFrame(columns=["date", "region", "sst"])
    processed = set()

buffer = []
errors = 0
n_done = len(processed)

# All year-month pairs in range, in chronological order
all_ym = []
y, m = START.year, START.month
while date(y, m, 1) <= END:
    all_ym.append((y, m))
    m += 1
    if m > 12:
        m, y = 1, y + 1

all_years  = list(range(START.year, END.year + 1))
total_days = (END - START).days + 1
print(f"Years: {len(all_years)}  |  Year-months: {len(all_ym)}  |  Calendar days: {total_days:,}")


# ── Helpers ───────────────────────────────────────────────────────────────────
def flush():
    global ckpt_df, buffer
    if buffer:
        ckpt_df = pd.concat([ckpt_df, pd.DataFrame(buffer)], ignore_index=True)
        ckpt_df.to_csv(CHECKPOINT, index=False)
        buffer = []


def fetch_one(d: date) -> bool:
    """Download, extract, delete one date. Returns True if newly processed."""
    global buffer, errors, n_done
    date_str = d.isoformat()
    if date_str in processed:
        return False

    nc_path = DATA_DIR / f"oisst-avhrr-v02r01.{d.strftime('%Y%m%d')}.nc"
    try:
        urllib.request.urlretrieve(make_url(d), nc_path)
    except Exception:
        errors += 1
        return False

    try:
        rows = extract_regions(nc_path, date_str)
        buffer.extend(rows)
        processed.add(date_str)
        n_done += 1
        return True
    except Exception as e:
        errors += 1
        print(f"  ERROR {date_str}: {e}")
        return False
    finally:
        nc_path.unlink(missing_ok=True)


# ── Phase 1: one file per year ────────────────────────────────────────────────
print("\n━━━ Phase 1: one file per year ━━━")
n1 = 0
for year in all_years:
    # July 15 is mid-year; clamp to available range for edge years
    candidate = date(year, 7, 15)
    if candidate < START:
        candidate = START           # 1981 starts Sept 1
    elif candidate > END:
        candidate = date(year, 1, 15)  # current year: use Jan instead

    if fetch_one(candidate):
        n1 += 1
        print(f"  {candidate}  [{n1}/{len(all_years)}]  total={n_done}  errors={errors}")
    if n1 % 10 == 0:
        flush()

flush()
print(f"Phase 1 complete — {n1} new dates\n")


# ── Phase 2: one file per month (the 15th) ────────────────────────────────────
print("━━━ Phase 2: one file per month ━━━")
n2 = 0
for i, (year, month) in enumerate(all_ym):
    _, days_in_month = monthrange(year, month)
    d = date(year, month, min(15, days_in_month))
    if d < START or d > END:
        continue
    if fetch_one(d):
        n2 += 1
    if (i + 1) % 10 == 0:
        flush()
        print(f"  {year}-{month:02d}-15  [{i+1}/{len(all_ym)}]  new={n2}  total={n_done}  errors={errors}")

flush()
print(f"Phase 2 complete — {n2} new dates\n")


# ── Phase 3: every remaining day, breadth-first ───────────────────────────────
print("━━━ Phase 3: all remaining days (breadth-first by day-of-month) ━━━")
n3 = 0
for day_offset in range(31):          # rounds 1–31 (the 1st through the 31st)
    round_new = 0
    for year, month in all_ym:
        day = day_offset + 1
        _, days_in_month = monthrange(year, month)
        if day > days_in_month:       # e.g. Feb has no 29th/30th/31st
            continue
        d = date(year, month, day)
        if d < START or d > END:
            continue
        if fetch_one(d):
            n3 += 1
            round_new += 1
            if n3 % 50 == 0:
                flush()

    flush()
    pct = 100 * n_done / total_days
    print(f"  Round {day_offset+1:>2}/31 — {round_new} new  |  total={n_done:,} ({pct:.1f}%)  errors={errors}")

print(f"\nPhase 3 complete — {n3} new dates")
print(f"\nAll phases done.  Total processed: {n_done:,}  |  Errors: {errors}")


Resuming — 637 dates already in checkpoint
Years: 46  |  Year-months: 537  |  Calendar days: 16,331

━━━ Phase 1: one file per year ━━━
Phase 1 complete — 0 new dates

━━━ Phase 2: one file per month ━━━
  1982-06-15  [10/537]  new=0  total=637  errors=0
  1983-04-15  [20/537]  new=0  total=637  errors=0
  1984-02-15  [30/537]  new=0  total=637  errors=0
  1984-12-15  [40/537]  new=0  total=637  errors=0
  1985-10-15  [50/537]  new=0  total=637  errors=0
  1986-08-15  [60/537]  new=0  total=637  errors=0
  1987-06-15  [70/537]  new=0  total=637  errors=0
  1988-04-15  [80/537]  new=0  total=637  errors=0
  1989-02-15  [90/537]  new=0  total=637  errors=0
  1989-12-15  [100/537]  new=0  total=637  errors=0
  1990-10-15  [110/537]  new=0  total=637  errors=0
  1991-08-15  [120/537]  new=0  total=637  errors=0
  1992-06-15  [130/537]  new=0  total=637  errors=0
  1993-04-15  [140/537]  new=0  total=637  errors=0
  1994-02-15  [150/537]  new=0  total=637  errors=0
  1994-12-15  [160/537]  

KeyboardInterrupt: 

## Aggregate to annual means and export

Groups the daily checkpoint by `(region, year)` and computes the mean SST, then writes the final CSV.  
Run this cell any time — even mid-pipeline — to see partial results while the download continues.


In [8]:
daily = pd.read_csv(CHECKPOINT)
daily["year"] = pd.to_datetime(daily["date"]).dt.year

annual = (
    daily
    .groupby(["region", "year"], sort=True)["sst"]
    .agg(mean_temp="mean", n_days="count")
    .reset_index()
)
annual["mean_temp"] = annual["mean_temp"].round(3)

print(annual.to_string(index=False))
print(f"\n{len(annual)} rows  |  {annual['year'].nunique()} years  |  {annual['region'].nunique()} regions")
print(f"Years with < 365 days of data: {annual[annual['n_days'] < 365][['region','year','n_days']].to_string(index=False)}")


              region  year  mean_temp  n_days
        Bay of Fundy  1981     10.661      20
        Bay of Fundy  1982      7.296      60
        Bay of Fundy  1983      7.792      60
        Bay of Fundy  1984      7.298      60
        Bay of Fundy  1985      6.932      60
        Bay of Fundy  1986      7.100      60
        Bay of Fundy  1987      6.845      60
        Bay of Fundy  1988      7.015      60
        Bay of Fundy  1989      7.081      50
        Bay of Fundy  1990      7.446      48
        Bay of Fundy  1991      7.445      48
        Bay of Fundy  1992      6.650      48
        Bay of Fundy  1993      6.930      48
        Bay of Fundy  1994      7.672      48
        Bay of Fundy  1995      7.143      48
        Bay of Fundy  1996      7.023      48
        Bay of Fundy  1997      7.193      48
        Bay of Fundy  1998      7.306      48
        Bay of Fundy  1999      8.046      48
        Bay of Fundy  2000      8.328      48
        Bay of Fundy  2001      7.

In [9]:
out = annual[["region", "year", "mean_temp"]]
out.to_csv(OUT_CSV, index=False)
print(f"Wrote {len(out)} rows → {OUT_CSV}")
print(out.head(10).to_string(index=False))


Wrote 138 rows → Data/sst_regions.csv
      region  year  mean_temp
Bay of Fundy  1981     10.661
Bay of Fundy  1982      7.296
Bay of Fundy  1983      7.792
Bay of Fundy  1984      7.298
Bay of Fundy  1985      6.932
Bay of Fundy  1986      7.100
Bay of Fundy  1987      6.845
Bay of Fundy  1988      7.015
Bay of Fundy  1989      7.081
Bay of Fundy  1990      7.446
